In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.impute import SimpleImputer

In [2]:
#load dataset
df = pd.read_csv("D:\\New Csv folder\\World University Rankings.csv")
df.head(3)

,name,scores_teaching,scores_research,scores_citations,scores_industry_income,scores_international_outlook,record_type,member_level,location,stats_number_students,stats_student_staff_ratio,stats_pc_intl_students,stats_female_male_ratio,subjects_offered,closed,unaccredited,overall_score
0,University of Oxford,96.6,100.0,99.0,98.7,97.5,master_account,0,United Kingdom,"21,750",10.9,42%,49:51:00,"Geography,Chemistry,Chemical Engineering,Biolo...",False,False,98.5
1,Stanford University,99.0,97.8,99.6,100.0,87.0,private,0,United States,"14,517",6.4,23%,47:53:00,"Computer Science,Communication & Media Studies...",False,False,98
2,Massachusetts Institute of Technology,98.6,96.2,99.7,100.0,93.8,private,0,United States,"11,085",8.0,33%,41:59:00,"Architecture,Economics & Econometrics,Archaeol...",False,False,97.9


In [3]:
df.isnull().sum()

name                              0
scores_teaching                 769
scores_research                 769
scores_citations                769
scores_industry_income          769
scores_international_outlook    769
record_type                       0
member_level                      0
location                          0
stats_number_students             0
stats_student_staff_ratio         0
stats_pc_intl_students            0
stats_female_male_ratio          93
subjects_offered                  4
closed                            0
unaccredited                      0
overall_score                   769
dtype: int64

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2673 entries, 0 to 2672
Data columns (total 17 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   name                          2673 non-null   object 
 1   scores_teaching               1904 non-null   float64
 2   scores_research               1904 non-null   float64
 3   scores_citations              1904 non-null   float64
 4   scores_industry_income        1904 non-null   float64
 5   scores_international_outlook  1904 non-null   float64
 6   record_type                   2673 non-null   object 
 7   member_level                  2673 non-null   int64  
 8   location                      2673 non-null   object 
 9   stats_number_students         2673 non-null   object 
 10  stats_student_staff_ratio     2673 non-null   float64
 11  stats_pc_intl_students        2673 non-null   object 
 12  stats_female_male_ratio       2580 non-null   object 
 13  sub

In [5]:
df = df.drop(columns=['name', 'location','stats_female_male_ratio','subjects_offered'], axis=1)

## Converting Datatype to int

In [6]:
# Remove ',' values

df['stats_number_students'] = df['stats_number_students'].apply(lambda x:x.replace(',',"")if ',' in str(x) else x)
df['stats_number_students'] = df['stats_number_students'].apply(lambda x:float(x))

In [7]:
# Remove '%' and replace empty strings with NaN

df['stats_pc_intl_students'] = df['stats_pc_intl_students'].apply(lambda x: x.replace('%', '') if isinstance(x, str) and '%' in x else x)
df['stats_pc_intl_students'] = df['stats_pc_intl_students'].replace('', np.nan)

# Convert to float
df['stats_pc_intl_students'] = df['stats_pc_intl_students'].astype(float)
df['stats_pc_intl_students'] = df['stats_pc_intl_students'].fillna(df['stats_pc_intl_students'].mean())

## Droping Null Values from data

In [8]:
df = df.dropna(subset=[
    'scores_teaching',
    'scores_research',
    'scores_citations',
    'scores_industry_income',
    'scores_international_outlook'
])
df.isnull().sum()

scores_teaching                 0
scores_research                 0
scores_citations                0
scores_industry_income          0
scores_international_outlook    0
record_type                     0
member_level                    0
stats_number_students           0
stats_student_staff_ratio       0
stats_pc_intl_students          0
closed                          0
unaccredited                    0
overall_score                   0
dtype: int64

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1904 entries, 0 to 1903
Data columns (total 13 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   scores_teaching               1904 non-null   float64
 1   scores_research               1904 non-null   float64
 2   scores_citations              1904 non-null   float64
 3   scores_industry_income        1904 non-null   float64
 4   scores_international_outlook  1904 non-null   float64
 5   record_type                   1904 non-null   object 
 6   member_level                  1904 non-null   int64  
 7   stats_number_students         1904 non-null   float64
 8   stats_student_staff_ratio     1904 non-null   float64
 9   stats_pc_intl_students        1904 non-null   float64
 10  closed                        1904 non-null   bool   
 11  unaccredited                  1904 non-null   bool   
 12  overall_score                 1904 non-null   object 
dtypes: bool(

## Outliers removing

In [10]:
# cols = ['scores_teaching','scores_research','scores_citations','scores_industry_income','scores_international_outlook','member_level',
#         'stats_number_students','stats_student_staff_ratio','stats_pc_intl_students']

# for col in cols:
#     Q1 = np.percentile(df[col],25)
#     Q3 = np.percentile(df[col],75)
#     IQR =  Q3 - Q1
#     lower  = Q1 - 1.5 * IQR
#     upper = Q3 + 1.5 * IQR
#     df = df[(df[col] > lower)|(df[col] < upper)]
# df = df.reset_index(drop=True)

## KNN model

In [11]:
def convert_score(val):
    if isinstance(val, str) and '–' in val:
        try:
            low, high = val.split('–')
            return (float(low) + float(high)) / 2
        except:
            return None  # or np.nan
    try:
        return float(val)
    except:
        return None  # or np.nan

df['overall_score'] = df['overall_score'].apply(convert_score)

In [12]:
le = LabelEncoder()
df['record_type'] = le.fit_transform(df['record_type'])
df

,scores_teaching,scores_research,scores_citations,scores_industry_income,scores_international_outlook,record_type,member_level,stats_number_students,stats_student_staff_ratio,stats_pc_intl_students,closed,unaccredited,overall_score
0,96.6,100.0,99.0,98.7,97.5,0,0,21750.0,10.9,42.0,False,False,98.5
1,99.0,97.8,99.6,100.0,87.0,1,0,14517.0,6.4,23.0,False,False,98.0
2,98.6,96.2,99.7,100.0,93.8,1,0,11085.0,8.0,33.0,False,False,97.9
3,97.7,99.9,99.4,84.2,90.8,1,0,20050.0,9.0,25.0,False,False,97.8
4,95.8,100.0,98.0,87.9,97.4,0,0,20565.0,11.5,38.0,False,False,97.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1899,19.5,11.7,34.5,36.0,20.4,0,0,39191.0,13.4,1.0,False,False,16.2
1900,22.0,12.4,17.6,59.3,38.8,0,0,17378.0,27.5,6.0,False,False,16.2
1901,23.9,8.6,26.8,16.4,38.1,0,0,13838.0,8.0,2.0,False,False,16.2
1902,16.9,10.5,28.8,23.1,30.6,0,0,24073.0,16.3,1.0,False,False,16.2


In [13]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scal = [
    'scores_teaching',
    'scores_research',
    'scores_citations',
    'scores_industry_income',
    'scores_international_outlook',
    'stats_number_students',
    'stats_student_staff_ratio',
    'stats_pc_intl_students'
]

df[scal] = scaler.fit_transform(df[scal])

In [14]:
df.isnull().sum()

scores_teaching                 0
scores_research                 0
scores_citations                0
scores_industry_income          0
scores_international_outlook    0
record_type                     0
member_level                    0
stats_number_students           0
stats_student_staff_ratio       0
stats_pc_intl_students          0
closed                          0
unaccredited                    0
overall_score                   0
dtype: int64

In [15]:
X = df.drop(['overall_score'],axis=1)
Y = df['overall_score']
x_train, x_test, y_train, y_test = train_test_split(X,Y,test_size=0.2,random_state=42)

In [16]:
model = KNeighborsRegressor(n_neighbors=5)
model.fit(x_train,y_train)

,n_neighbors,5
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [17]:
y_pred = model.predict(x_test)

In [18]:
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)  
r2 = r2_score(y_test, y_pred)

print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R² Score: {r2:.4f}")

MSE: 10.5643
RMSE: 3.2503
R² Score: 0.9607


In [19]:
values = [4.836838,	4.587633,	1.867574,	1.975345,	2.180906,	0,	0,	-0.017464,	-0.702042,	2.670829,	False,	False	]

prediction = model.predict([values])
print("Predicted value:", prediction)

Predicted value: [96.7]


C:\Users\SART\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but KNeighborsRegressor was fitted with feature names
  warnings.warn(
